# Colab Enterprise Workspace Check

과제 Runtime이 올바른 사용자 권한, BigQuery Dataset, GCS Workspace를 사용하는지 확인하는 smoke-test notebook입니다.

In [ ]:
import os

TASK_ID = os.environ.get("DATA_MODEL_TASK_ID")
DATASET = os.environ.get("DATA_MODEL_DATASET")
WORKSPACE_BUCKET = os.environ.get("DATA_MODEL_WORKSPACE_BUCKET")

print("TASK_ID          =", TASK_ID)
print("DATASET          =", DATASET)
print("WORKSPACE_BUCKET =", WORKSPACE_BUCKET)

In [ ]:
import google.auth

credentials, detected_project = google.auth.default()
print("Detected project =", detected_project)
print("Credential type  =", type(credentials).__name__)

In [ ]:
from google.cloud import bigquery

if not DATASET:
    raise RuntimeError("DATA_MODEL_DATASET is not set")

dataset_project, dataset_id = DATASET.split(".", 1)
client = bigquery.Client(project=dataset_project)
dataset = client.get_dataset(f"{dataset_project}.{dataset_id}")
print("BigQuery dataset OK:", dataset.full_dataset_id)

In [ ]:
from google.cloud import storage

if not WORKSPACE_BUCKET or not WORKSPACE_BUCKET.startswith("gs://"):
    raise RuntimeError("DATA_MODEL_WORKSPACE_BUCKET is not set")

bucket_name = WORKSPACE_BUCKET.removeprefix("gs://")
storage_client = storage.Client()
bucket = storage_client.bucket(bucket_name)
blob = bucket.blob(f"smoke-test/{TASK_ID or 'unknown'}.txt")
blob.upload_from_string("Colab Enterprise workspace access OK\n")
print("GCS write OK: gs://" + bucket_name + "/" + blob.name)